# Sec 2e — Single-trial exemplars

Per-trial, per-session demonstrations of true signal vs. PSID/DPAD/VARMA predictions.

* Figs 18-21 — neural reconstruction time series (side-by-side DBS-OFF / DBS-ON, 4 sessions)
* Figs 29-36 — neural forecast exemplars (best trial per condition x 4 sessions)
* Figs 50-55 — 4x3 exemplar grids (recon / forecast x RMSE / Pearson / VAF)

In [ ]:
import sys, os

os.chdir("/home/bobby/repos/latent-neural-dynamics-modeling")
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from modules.style import (
    COLOR_PSID,
    COLOR_DPAD,
    COLOR_VARMA,
    COLOR_TRUE,
    COLOR_DBS_OFF,
    COLOR_DBS_ON,
    apply_modules.style,
    panel_label,
    hex_to_rgba,
)
from modules.loaders import (
    discover_session_run,
    EXP_BEHAVIORAL,
    EXP_NEURAL,
    SESSIONS,
)
from modules.specs import (
    ThesisNeuralTimeseriesSpec,
    ThesisC2ForecastSpec,
)

OUT = Path("thesis_figures/sec2")
OUT.mkdir(parents=True, exist_ok=True)
results_root = Path("results").resolve()

# --- Inlined exemplar specs (formerly modules.sec2_common.py) ---
_TRIAL_INDICES = {
    "PDI1_S2": (11, 27),
    "PDI1_S4": (17, 5),
    "PDI4_S2": (5, 18),
    "PDI4_S3": (9, 25),
}
THESIS_NEURAL_TIMESERIES = []
THESIS_C2_FORECASTS = []
for _session, (_off, _on) in _TRIAL_INDICES.items():
    _psid_var, _psid_ts = discover_session_run(results_root, "psid", EXP_BEHAVIORAL, _session)
    _dpad_var, _dpad_ts = discover_session_run(results_root, "dpad", EXP_BEHAVIORAL, _session)
    _varma_var, _varma_ts = discover_session_run(results_root, "varma", EXP_BEHAVIORAL, _session)
    _, _varma_ts_off = discover_session_run(results_root, "varma", EXP_BEHAVIORAL, _session, "dbs_off")
    _, _varma_ts_on = discover_session_run(results_root, "varma", EXP_BEHAVIORAL, _session, "dbs_on")
    if not _psid_ts or not _varma_ts:
        continue
    THESIS_NEURAL_TIMESERIES.append(ThesisNeuralTimeseriesSpec(
        section_title=_session.replace("_", " "),
        participant_label=_session.split("_")[0],
        psid_variant=_psid_var,
        dpad_variant=_dpad_var,
        varma_variant=_varma_var,
        psid_run_ts=_psid_ts,
        dpad_run_ts=_dpad_ts,
        varma_run_ts=_varma_ts,
        split="test",
        trial_idx_off=_off,
        trial_idx_on=_on,
        neural_y_channel_idx=5,
        neural_y_feature_name="ECOG_1_beta_27_30_raw",
        use_adjacent_off_on_trials=True,
        exemplar_layout="side_by_side",
        varma_run_ts_off=_varma_ts_off or None,
        varma_run_ts_on=_varma_ts_on or None,
    ))
    THESIS_C2_FORECASTS.append(ThesisC2ForecastSpec(
        section_title=_session,
        participant_label=_session.split("_")[0],
        psid_variant=_psid_var,
        dpad_variant=_dpad_var,
        varma_variant=_varma_var,
        psid_run_ts=_psid_ts,
        dpad_run_ts=_dpad_ts,
        varma_run_ts=_varma_ts,
        split="test",
        trial_idx_off=_off,
        trial_idx_on=_on,
        channel_idx=5,
        forecast_target="Y",
        neural_y_feature_name="ECOG_1_beta_27_30_raw",
        varma_run_ts_off=_varma_ts_off or None,
        varma_run_ts_on=_varma_ts_on or None,
    ))

from modules.lib.aggregate_rmse import _key_index_map, _trial_key, normalize_stim
from modules.loaders import (
    load_split_results,
    load_split_results_required,
)
from modules.lib.loaders import (
    load_precomputed_results,
    channels_as_str_list,
    resolve_neural_y_channel_idx,
    neural_y_feature_label,
)

from types import SimpleNamespace as _NS


def _session_ns(session, exp_type):
    pv, pt = discover_session_run(results_root, "psid", exp_type, session)
    vv, vt = discover_session_run(results_root, "varma", exp_type, session)
    dv, dt = discover_session_run(results_root, "dpad", exp_type, session)
    return _NS(
        psid_variant=pv or "", psid_run_ts=pt or "",
        varma_variant=vv or "", varma_run_ts=vt or "",
        dpad_variant=dv or "", dpad_run_ts=dt or "", label=session,
    )


_all_session_objs = [_session_ns(s, EXP_BEHAVIORAL) for s in SESSIONS]
_all_lap_session_objs = [_session_ns(s, EXP_NEURAL) for s in SESSIONS]

from types import SimpleNamespace as _NS


def _session_ns(session, exp_type):
    pv, pt = discover_session_run(results_root, "psid", exp_type, session)
    vv, vt = discover_session_run(results_root, "varma", exp_type, session)
    dv, dt = discover_session_run(results_root, "dpad", exp_type, session)
    return _NS(
        psid_variant=pv or "", psid_run_ts=pt or "",
        varma_variant=vv or "", varma_run_ts=vt or "",
        dpad_variant=dv or "", dpad_run_ts=dt or "", label=session,
    )


_all_session_objs = [_session_ns(s, EXP_BEHAVIORAL) for s in SESSIONS]
_all_lap_session_objs = [_session_ns(s, EXP_NEURAL) for s in SESSIONS]

apply_modules.style()

## Figs 18-21: Neural reconstruction time series

Side-by-side panels (DBS-OFF | DBS-ON) per session: true `ECOG_1_beta_27_30_raw`
vs. PSID/DPAD/VARMA one-step prediction. Inlined matplotlib builder; data
loaders remain imports from `thesis_lib`.

In [ ]:
def _consec_pair_y(res_p, ch_idx):
    """Best adjacent OFF/ON pair scored by sum of Y Pearson r."""
    Y, Yp = res_p.get("Y", []), res_p.get("Yp", [])
    stims = res_p.get("stim", [])
    best, best_score = (None, None), -np.inf
    for i in range(len(stims) - 1):
        si = normalize_stim(stims[i])
        sj = normalize_stim(stims[i + 1])
        if si is None or sj is None or si == sj:
            continue
        off_i = i if si == "off" else i + 1
        on_i = i if si == "on" else i + 1
        score = 0.0
        for ti in (off_i, on_i):
            if ti >= len(Y) or ti >= len(Yp):
                continue
            y = np.asarray(Y[ti], dtype=float)
            yp = np.asarray(Yp[ti], dtype=float)
            ci = min(ch_idx, y.shape[1] - 1) if y.ndim == 2 else 0
            yc = y[:, ci] if y.ndim == 2 else y.ravel()
            ypc = yp[:, ci] if yp.ndim == 2 else yp.ravel()
            n = min(len(yc), len(ypc))
            if n < 10 or yc[:n].std() < 1e-9 or ypc[:n].std() < 1e-9:
                continue
            r = float(np.corrcoef(yc[:n], ypc[:n])[0, 1])
            if np.isfinite(r):
                score += r
        if score > best_score:
            best_score, best = score, (off_i, on_i)
    return best  # (off_idx, on_idx) or (None, None)


def _consec_pair_z(res_p, ch_idx):
    """Best adjacent OFF/ON pair scored by sum of Z Pearson r."""
    Z, Zp = res_p.get("Z", []), res_p.get("Zp", [])
    stims = res_p.get("stim", [])
    best, best_score = (None, None), -np.inf
    for i in range(len(stims) - 1):
        si = normalize_stim(stims[i])
        sj = normalize_stim(stims[i + 1])
        if si is None or sj is None or si == sj:
            continue
        off_i = i if si == "off" else i + 1
        on_i = i if si == "on" else i + 1
        score = 0.0
        for ti in (off_i, on_i):
            if ti >= len(Z) or ti >= len(Zp):
                continue
            z = np.asarray(Z[ti], dtype=float)
            zp = np.asarray(Zp[ti], dtype=float)
            ci = min(ch_idx, z.shape[1] - 1) if z.ndim == 2 else 0
            zc = z[:, ci] if z.ndim == 2 else z.ravel()
            zpc = zp[:, ci] if zp.ndim == 2 else zp.ravel()
            n = min(len(zc), len(zpc))
            if n < 10 or zc[:n].std() < 1e-9 or zpc[:n].std() < 1e-9:
                continue
            r = float(np.corrcoef(zc[:n], zpc[:n])[0, 1])
            if np.isfinite(r):
                score += r
        if score > best_score:
            best_score, best = score, (off_i, on_i)
    return best

In [ ]:
from modules.lib.compose import _session_mean_rmse_y_triplet
from modules.lib.exemplar_trials import (
    find_best_trial_indices_per_condition,
    find_best_channel_and_trial,
    resolve_off_on_indices_from_spec,
)
from modules.lib.loaders import (
    extract_trial_y_series,
    thesis_exemplar_tagline,
    ThesisDataError,
)
from modules.lib.specs import infer_varma_off_on_run_ts
from modules.lib.transforms import rmse_z, z_true_and_preds

EXEMPLAR_METRIC = "rmse"
AUTO_BEST_CHANNEL = False


def _slice_trial_tail(t_abs, seg_s, z_true, z_psid, z_dpad, z_varma):
    t = np.asarray(t_abs, dtype=float).ravel()
    if t.size == 0:
        e = np.array([], dtype=float)
        return e, e, e, e, e
    t_hi = float(np.nanmax(t))
    t_lo = t_hi - float(seg_s)
    m = t >= t_lo
    arrays = [
        np.asarray(a, dtype=float).ravel() for a in (z_true, z_psid, z_dpad, z_varma)
    ]
    return (t[m],) + tuple(a[m] for a in arrays)


def _mpl_side_by_side_exemplar(
    panel_off,
    panel_on,
    y_axis_label="z-score",
    *,
    channel_label="",
    session_label="",
    segment_s=1.0,
):
    def _prep(p):
        t_raw = np.asarray(p["t_abs"], dtype=float)
        t_sl, zt, zp, zd, zv = _slice_trial_tail(
            t_raw, segment_s, p["z_true"], p["z_psid"], p["z_dpad"], p["z_varma"]
        )
        return t_sl, zt, zp, zd, zv

    to, zto, zpo, zdo, zvo = _prep(panel_off)
    tn, ztn, zpn, zdn, zvn = _prep(panel_on)

    fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.2), sharey=True)
    for ax, t, zt, zp, zd, zv, panel, letter, title in (
        (axes[0], to, zto, zpo, zdo, zvo, panel_off, "A", "DBS-OFF"),
        (axes[1], tn, ztn, zpn, zdn, zvn, panel_on, "B", "DBS-ON"),
    ):
        if t.size == 0:
            panel_label(
                ax,
                letter,
                (
                    f"{session_label} — {title} — {channel_label}"
                    if channel_label
                    else f"{session_label} — {title}"
                ),
            )
            continue
        for y_hat, half, col in (
            (zp, panel["band_rmse_psid"], COLOR_PSID),
            (zd, panel["band_rmse_dpad"], COLOR_DPAD),
            (zv, panel["band_rmse_varma"], COLOR_VARMA),
        ):
            if half is None or not np.isfinite(half) or half <= 0:
                continue
            yh = np.asarray(y_hat, dtype=float)
            if len(t) != len(yh) or np.all(np.isnan(yh)):
                continue
            ax.fill_between(t, yh - half, yh + half, color=col, alpha=0.15, linewidth=0)
        ax.plot(t, zt, color=COLOR_TRUE, linewidth=1.4, label="y_true")
        if not np.all(np.isnan(zp)):
            ax.plot(t, zp, color=COLOR_PSID, linewidth=1.2, label="PSID")
        if not np.all(np.isnan(zd)):
            ax.plot(
                t, zd, color=COLOR_DPAD, linewidth=1.2, linestyle="--", label="DPAD"
            )
        if not np.all(np.isnan(zv)):
            ax.plot(
                t, zv, color=COLOR_VARMA, linewidth=1.2, linestyle=":", label="VARMA"
            )
        ax.set_xlabel("trial time (s)")
        panel_label(
            ax,
            letter,
            (
                f"{session_label} — {title} — {channel_label}"
                if channel_label
                else f"{session_label} — {title}"
            ),
        )
    axes[0].set_ylabel(y_axis_label)
    axes[0].legend()
    return fig


def compose_thesis_neural_figure(spec, results_root, _force_indices=None):
    res_p = load_split_results_required(
        results_root, spec.psid_variant, spec.psid_run_ts, spec.split
    )
    res_d = load_split_results(
        results_root, spec.dpad_variant, spec.dpad_run_ts, spec.split
    )
    res_v = load_split_results_required(
        results_root, spec.varma_variant, spec.varma_run_ts, spec.split
    )

    def _n_y_channels(res):
        if res is None or not res.get("Y"):
            return None
        arr = np.asarray(res["Y"][0])
        return int(arr.shape[1]) if arr.ndim == 2 else 1

    n_caps = [
        c
        for c in (_n_y_channels(res_p), _n_y_channels(res_d), _n_y_channels(res_v))
        if c
    ]
    n_cap = min(n_caps) if n_caps else None
    ch_idx = spec.neural_y_channel_idx

    if _force_indices is not None:
        i_off, i_on = _force_indices
    elif AUTO_BEST_CHANNEL:
        ch_trial = find_best_channel_and_trial(
            res_p, input_mode="neural", n_channels=n_cap
        )
        if ch_trial is not None:
            ch_off, i_off, ch_on, i_on = ch_trial
            ch_idx = ch_off
        else:
            i_off, i_on = resolve_off_on_indices_from_spec(
                trial_idx_off=spec.trial_idx_off,
                trial_idx_on=spec.trial_idx_on,
                use_adjacent_off_on_trials=spec.use_adjacent_off_on_trials,
                split_res=res_p,
            )
    else:
        best_pair = find_best_trial_indices_per_condition(
            res_p, channel_idx=ch_idx, input_mode="neural"
        )
        if best_pair is not None:
            i_off, i_on = best_pair
        else:
            i_off, i_on = resolve_off_on_indices_from_spec(
                trial_idx_off=spec.trial_idx_off,
                trial_idx_on=spec.trial_idx_on,
                use_adjacent_off_on_trials=spec.use_adjacent_off_on_trials,
                split_res=res_p,
            )

    res_v_off = res_v_on = None
    if "dbs_both" in spec.varma_variant:
        v_off = spec.varma_variant.replace("dbs_both", "dbs_off")
        v_on = spec.varma_variant.replace("dbs_both", "dbs_on")
        ts_off, ts_on = spec.varma_run_ts_off, spec.varma_run_ts_on
        if ts_off is None or ts_on is None:
            inf = infer_varma_off_on_run_ts(spec.varma_variant, spec.varma_run_ts)
            if inf is not None:
                ts_off, ts_on = inf
        if ts_off is None or ts_on is None:
            raise ThesisDataError(
                f"varma_run_ts_off/on required for {spec.section_title!r}"
            )
        res_v_off = load_split_results_required(results_root, v_off, ts_off, spec.split)
        res_v_on = load_split_results_required(results_root, v_on, ts_on, spec.split)

    def _varma_res_and_idx(panel, psid_trial_idx):
        rv = res_v_off if panel == "off" else res_v_on
        if rv is not None:
            k = _trial_key(res_p, psid_trial_idx)
            mp = _key_index_map(rv)
            if k in mp:
                return rv, mp[k]
        return (
            (res_v, psid_trial_idx)
            if res_v is not None
            else ({"Y": [], "Yp": []}, psid_trial_idx)
        )

    res_v_off_use, idx_v_off = _varma_res_and_idx("off", i_off)
    res_v_on_use, idx_v_on = _varma_res_and_idx("on", i_on)
    map_v_off = _key_index_map(res_v_off_use)
    map_v_on = _key_index_map(res_v_on_use)

    band_p_off, band_d_off, band_v_off = _session_mean_rmse_y_triplet(
        res_p, res_d, res_v_off_use, map_v_off, i_off, ch_idx, "off"
    )
    band_p_on, band_d_on, band_v_on = _session_mean_rmse_y_triplet(
        res_p, res_d, res_v_on_use, map_v_on, i_on, ch_idx, "on"
    )

    off = extract_trial_y_series(
        res_p, res_d, res_v_off_use, i_off, ch_idx, varma_trial_idx=idx_v_off
    )
    on = extract_trial_y_series(
        res_p, res_d, res_v_on_use, i_on, ch_idx, varma_trial_idx=idx_v_on
    )

    zt_o, zp_o, zd_o, zv_o = z_true_and_preds(
        off.z_true_raw, off.z_psid, off.z_dpad, off.z_varma
    )
    zt_n, zp_n, zd_n, zv_n = z_true_and_preds(
        on.z_true_raw, on.z_psid, on.z_dpad, on.z_varma
    )

    panel_off = dict(
        t_abs=off.t_abs,
        z_true=zt_o,
        z_psid=zp_o,
        z_dpad=zd_o,
        z_varma=zv_o,
        band_rmse_psid=band_p_off,
        band_rmse_dpad=band_d_off,
        band_rmse_varma=band_v_off,
    )
    panel_on = dict(
        t_abs=on.t_abs,
        z_true=zt_n,
        z_psid=zp_n,
        z_dpad=zd_n,
        z_varma=zv_n,
        band_rmse_psid=band_p_on,
        band_rmse_dpad=band_d_on,
        band_rmse_varma=band_v_on,
    )

    y_meta = neural_y_feature_label(
        res_p, ch_idx, neural_y_feature_name=spec.neural_y_feature_name
    )
    caption = thesis_exemplar_tagline(
        res_p, i_off, i_on, y_meta, participant_label=spec.participant_label
    )
    if (spec.caption_extra or "").strip():
        caption = f"{caption} · {spec.caption_extra.strip()}"

    fig = _mpl_side_by_side_exemplar(
        panel_off,
        panel_on,
        y_axis_label="z-score",
        channel_label=y_meta,
        session_label=spec.section_title,
        segment_s=spec.exemplar_mid_segment_s,
    )
    return fig, caption


# ── Figs 18-21: Y reconstruction, EXP_BEHAVIORAL ──────────────────────────────
fig_num = 18
for n_spec in THESIS_NEURAL_TIMESERIES:
    res_p_tmp = load_split_results_required(
        results_root, n_spec.psid_variant, n_spec.psid_run_ts, n_spec.split
    )
    pair = _consec_pair_y(res_p_tmp, n_spec.neural_y_channel_idx)
    fig, cap = compose_thesis_neural_figure(n_spec, results_root, _force_indices=pair)
    fname = f"fig_{fig_num:03d}_neural_ts_{n_spec.section_title.replace(' ','_')}.png"
    fig.savefig(str(OUT / fname))
    plt.show()
    print(
        f"Fig {fig_num}: Y recon (EXP_BEHAVIORAL) — {n_spec.section_title} ('{n_spec.neural_y_feature_name}'). Consecutive pair: OFF={pair[0]}, ON={pair[1]}. {cap}"
    )
    fig_num += 1

In [ ]:
# ── Figs 22-25: Y reconstruction, EXP_NEURAL (ECoG self-recon from ECoG+LFP model) ──
THESIS_NEURAL_TIMESERIES_LAP = []
for _session, (_off, _on) in _TRIAL_INDICES.items():
    _pv, _pt = discover_session_run(results_root, "psid", EXP_NEURAL, _session)
    _dv, _dt = discover_session_run(results_root, "dpad", EXP_NEURAL, _session)
    _vv, _vt = discover_session_run(results_root, "varma", EXP_NEURAL, _session)
    _, _vt_off = discover_session_run(
        results_root, "varma", EXP_NEURAL, _session, "dbs_off"
    )
    _, _vt_on = discover_session_run(
        results_root, "varma", EXP_NEURAL, _session, "dbs_on"
    )
    if not _pt or not _vt:
        continue
    THESIS_NEURAL_TIMESERIES_LAP.append(
        ThesisNeuralTimeseriesSpec(
            section_title=_session.replace("_", " "),
            participant_label=_session.split("_")[0],
            psid_variant=_pv,
            dpad_variant=_dv,
            varma_variant=_vv,
            psid_run_ts=_pt,
            dpad_run_ts=_dt,
            varma_run_ts=_vt,
            split="test",
            trial_idx_off=_off,
            trial_idx_on=_on,
            neural_y_channel_idx=5,
            neural_y_feature_name="ECOG_1_beta_27_30_raw",
            use_adjacent_off_on_trials=True,
            exemplar_layout="side_by_side",
            varma_run_ts_off=_vt_off or None,
            varma_run_ts_on=_vt_on or None,
        )
    )

fig_num = 22
for n_spec in THESIS_NEURAL_TIMESERIES_LAP:
    res_p_tmp = load_split_results_required(
        results_root, n_spec.psid_variant, n_spec.psid_run_ts, n_spec.split
    )
    pair = _consec_pair_y(res_p_tmp, n_spec.neural_y_channel_idx)
    fig, cap = compose_thesis_neural_figure(n_spec, results_root, _force_indices=pair)
    fname = (
        f"fig_{fig_num:03d}_neural_ts_lap_{n_spec.section_title.replace(' ','_')}.png"
    )
    fig.savefig(str(OUT / fname))
    plt.show()
    print(
        f"Fig {fig_num}: Y recon (EXP_NEURAL) — {n_spec.section_title}. Consecutive pair: OFF={pair[0]}, ON={pair[1]}. {cap}"
    )
    fig_num += 1

In [ ]:
from modules.lib.loaders import get_trial_time_axis, transpose_if_needed
from modules.utils import trial_metric_z_for_model
import polars as _pl


def _z_feat_name(sess_ns, ch_idx, results_root):
    """Resolve actual Z feature name from PSID train parquet."""
    try:
        fdir = results_root / "psid" / sess_ns.psid_variant / "train"
        pqs = list(fdir.glob("test_results_*.parquet"))
        if not pqs:
            return f"Z ch{ch_idx}"
        df = _pl.read_parquet(str(pqs[0]))
        if "Z_features" not in df.columns:
            return f"Z ch{ch_idx}"
        names = df[0]["Z_features"][0].to_list()
        return names[ch_idx] if ch_idx < len(names) else f"Z ch{ch_idx}"
    except Exception:
        return f"Z ch{ch_idx}"


def _get_z_channel(res, trial_idx, ch_idx):
    """Extract (t_abs, z_true, z_pred) for one Z channel from split results."""
    Z = res.get("Z", [])
    Zp = res.get("Zp", [])
    if trial_idx >= len(Z):
        return None, None, None
    z = np.asarray(Z[trial_idx], dtype=float)
    zp = (
        np.asarray(Zp[trial_idx], dtype=float)
        if trial_idx < len(Zp)
        else np.full(len(z), np.nan)
    )
    if z.ndim == 2:
        ci = min(ch_idx, z.shape[1] - 1)
        z = z[:, ci]
        zp = zp[:, ci] if zp.ndim == 2 else zp.ravel()
    n = len(z)
    t_list = res.get("t_abs") or []
    t = (
        np.asarray(t_list[trial_idx], dtype=float)
        if trial_idx < len(t_list)
        else np.arange(n) / 200.0
    )
    return t, z, zp


def _compose_z_reconstruction(sess_ns, ch_idx, exp_label, fig_num):
    """Build side-by-side OFF|ON Z reconstruction exemplar. Uses consecutive pair selection."""
    if not sess_ns.psid_run_ts:
        print(f"Fig {fig_num}: SKIP {sess_ns.label} — no PSID variant")
        return None
    res_p = load_split_results_required(
        results_root, sess_ns.psid_variant, sess_ns.psid_run_ts, "test"
    )
    res_d = load_split_results(
        results_root, sess_ns.dpad_variant, sess_ns.dpad_run_ts, "test"
    )
    feat_name = _z_feat_name(sess_ns, ch_idx, results_root)

    pair = _consec_pair_z(res_p, ch_idx)
    if pair[0] is None:
        print(f"Fig {fig_num}: SKIP {sess_ns.label} — no consecutive pair found")
        return None
    off_i, on_i = pair

    res_v_off = res_v_on = None
    if sess_ns.varma_run_ts and "dbs_both" in sess_ns.varma_variant:
        vv_off = sess_ns.varma_variant.replace("dbs_both", "dbs_off")
        vv_on = sess_ns.varma_variant.replace("dbs_both", "dbs_on")
        _, _vt_off = discover_session_run(
            results_root, "varma", exp_label, sess_ns.label, "dbs_off"
        )
        _, _vt_on = discover_session_run(
            results_root, "varma", exp_label, sess_ns.label, "dbs_on"
        )
        if _vt_off:
            res_v_off = load_split_results_required(
                results_root, vv_off, _vt_off, "test"
            )
        if _vt_on:
            res_v_on = load_split_results_required(results_root, vv_on, _vt_on, "test")
    res_v_both = (
        load_split_results_required(
            results_root, sess_ns.varma_variant, sess_ns.varma_run_ts, "test"
        )
        if sess_ns.varma_run_ts
        else None
    )

    panels = {}
    for stim, cond_lbl, i_use in (("off", "OFF", off_i), ("on", "ON", on_i)):
        t, z_true, z_psid = _get_z_channel(res_p, i_use, ch_idx)
        _, _, z_dpad = (
            _get_z_channel(res_d, i_use, ch_idx)
            if res_d
            else (None, None, np.full_like(z_true, np.nan))
        )
        res_v_use = (res_v_off if stim == "off" else res_v_on) or res_v_both
        jv = None
        if res_v_use is not None:
            k = _trial_key(res_p, i_use)
            jv = _key_index_map(res_v_use).get(k)
        _, _, z_varma = (
            _get_z_channel(res_v_use, jv if jv is not None else i_use, ch_idx)
            if res_v_use
            else (None, None, np.full_like(z_true, np.nan))
        )
        if z_dpad is None:
            z_dpad = np.full_like(z_true, np.nan)
        if z_varma is None:
            z_varma = np.full_like(z_true, np.nan)
        panels[cond_lbl] = dict(
            t_abs=t,
            z_true=z_true,
            z_psid=z_psid,
            z_dpad=z_dpad,
            z_varma=z_varma,
            band_rmse_psid=None,
            band_rmse_dpad=None,
            band_rmse_varma=None,
        )
    return panels, off_i, on_i, feat_name


# ── Figs 37-40: Z reconstruction, EXP_NEURAL (Laplacian LFP ch0) ──────────────
_Z_LAP_CH = 0
fig_num = 37
for tri in _all_lap_session_objs:
    result = _compose_z_reconstruction(tri, _Z_LAP_CH, EXP_NEURAL, fig_num)
    if result is None:
        fig_num += 1
        continue
    panels, off_i, on_i, feat_name = result
    fig = _mpl_side_by_side_exemplar(
        panels["OFF"],
        panels["ON"],
        y_axis_label="z-score",
        channel_label=feat_name,
        session_label=tri.label,
    )
    fig.savefig(str(OUT / f"fig_{fig_num:03d}_z_recon_lap_{tri.label}.png"))
    plt.show()
    print(
        f"Fig {fig_num}: Z recon (Laplacian LFP, EXP_NEURAL) — {tri.label} ('{feat_name}'). Consecutive pair: OFF={off_i}, ON={on_i}."
    )
    fig_num += 1

# ── Figs 41-44: Z reconstruction, EXP_BEHAVIORAL (velocity X, ch0) ────────────
fig_num = 41
for tri in _all_session_objs:
    result = _compose_z_reconstruction(tri, 0, EXP_BEHAVIORAL, fig_num)
    if result is None:
        fig_num += 1
        continue
    panels, off_i, on_i, feat_name = result
    fig = _mpl_side_by_side_exemplar(
        panels["OFF"],
        panels["ON"],
        y_axis_label="z-score",
        channel_label=feat_name,
        session_label=tri.label,
    )
    fig.savefig(str(OUT / f"fig_{fig_num:03d}_z_recon_kin_{tri.label}.png"))
    plt.show()
    print(
        f"Fig {fig_num}: Z recon (kinematics, EXP_BEHAVIORAL) — {tri.label} ('{feat_name}'). Consecutive pair: OFF={off_i}, ON={on_i}."
    )
    fig_num += 1

## Figs 29-36: Neural forecast exemplars (per-session x OFF/ON)

Single-panel forecast exemplars — history (light blue) + forecast window (light red)
with true + PSID + VARMA traces. Best trial per (session, condition) selected by
minimising `max(rmse_psid, rmse_varma)` over the entire forecast window.

In [ ]:
from modules.lib.forecast_horizon_rmse import _per_step_abs_err_z_future

_FORECAST_CTX_ALPHA = 0.08
_FORECAST_FUT_ALPHA = 0.07


def _trial_forecast_rmse(res, k_true, k_pred, trial_idx, channel_idx):
    if res is None:
        return float("nan")
    zt = res.get(k_true)
    zp = res.get(k_pred)
    if zt is None or zp is None or trial_idx >= len(zt) or trial_idx >= len(zp):
        return float("nan")
    err = _per_step_abs_err_z_future(zt[trial_idx], zp[trial_idx], channel_idx)
    if err is None or err.size == 0:
        return float("nan")
    finite = err[np.isfinite(err)]
    return float(np.sqrt(np.mean(finite**2))) if finite.size else float("nan")


def _consec_pair_fc(res_p, k_true, k_pred, ch_idx):
    """Best adjacent OFF/ON pair by min max(rmse_off, rmse_on) in forecast results."""
    stims = res_p.get("stim", [])
    n = len(res_p.get(k_true) or [])
    best, best_score = (None, None), float("inf")
    for i in range(min(len(stims) - 1, n - 1)):
        si = normalize_stim(stims[i])
        sj = normalize_stim(stims[i + 1])
        if si is None or sj is None or si == sj:
            continue
        off_i = i if si == "off" else i + 1
        on_i = i if si == "on" else i + 1
        r_off = _trial_forecast_rmse(res_p, k_true, k_pred, off_i, ch_idx)
        r_on = _trial_forecast_rmse(res_p, k_true, k_pred, on_i, ch_idx)
        score = max(r_off, r_on)
        if np.isfinite(score) and score < best_score:
            best_score, best = score, (off_i, on_i)
    return best  # (off_idx, on_idx) or (None, None)


def _load_h5_forecast(results_root, variant, split="test", horizon="h5"):
    if not variant:
        return None
    fw = variant.split("_", 1)[0]
    fdir = results_root / fw / variant / "forecast" / horizon
    files = sorted((fdir / split).glob("test_results_*.parquet"))
    if not files:
        return None
    ts = files[0].name[len("test_results_") : -len(".parquet")]
    try:
        return load_precomputed_results(fdir, ts, split)
    except Exception:
        return None


def _resolve_varma_off_on(spec):
    res_p = load_split_results_required(
        results_root, spec.psid_variant, spec.psid_run_ts, "test"
    )
    res_v = load_split_results_required(
        results_root, spec.varma_variant, spec.varma_run_ts, "test"
    )
    res_v_off = res_v_on = None
    if "dbs_both" in spec.varma_variant:
        v_off_var = spec.varma_variant.replace("dbs_both", "dbs_off")
        v_on_var = spec.varma_variant.replace("dbs_both", "dbs_on")
        if spec.varma_run_ts_off:
            res_v_off = load_split_results_required(
                results_root, v_off_var, spec.varma_run_ts_off, "test"
            )
        if spec.varma_run_ts_on:
            res_v_on = load_split_results_required(
                results_root, v_on_var, spec.varma_run_ts_on, "test"
            )
    ch_use = resolve_neural_y_channel_idx(
        res_p, spec.neural_y_feature_name, spec.channel_idx
    )
    return res_p, res_v, res_v_off, res_v_on, ch_use


from modules.lib.c2_forecast_timeseries import _build_one_panel as _forecast_rowdata


def _mpl_forecast_panel(
    rowdata, neu_lbl, condition_label, *, channel_label="", session_label=""
):
    (t_full, z_true, z_psid, z_dpad, z_varma, _u, _l, _rp, _rd, _rv, n_hist) = rowdata
    t_full = np.asarray(t_full, dtype=float).ravel()
    n_hist = int(n_hist)

    def _gap(a):
        a = np.asarray(a, dtype=float).ravel()
        if 0 < n_hist < len(a):
            return np.concatenate([a[:n_hist], [np.nan], a[n_hist:]])
        return a

    t_plot = _gap(t_full)
    fig, ax = plt.subplots(figsize=(8.5, 3.2))

    if t_full.size and n_hist > 0:
        ax.axvspan(
            float(t_full[0]),
            float(t_full[n_hist - 1]),
            color=COLOR_DBS_OFF,
            alpha=_FORECAST_CTX_ALPHA,
            linewidth=0,
        )
    if t_full.size and n_hist < len(t_full):
        ax.axvspan(
            float(t_full[n_hist]),
            float(t_full[-1]),
            color=COLOR_DBS_ON,
            alpha=_FORECAST_FUT_ALPHA,
            linewidth=0,
        )
        ax.axvline(
            float(t_full[n_hist]),
            color="#444441",
            linewidth=1.0,
            linestyle="--",
            alpha=0.6,
        )

    ax.plot(t_plot, _gap(z_true), color=COLOR_TRUE, linewidth=1.4, label="y_true")
    if not np.all(np.isnan(z_psid)):
        ax.plot(t_plot, _gap(z_psid), color=COLOR_PSID, linewidth=1.2, label="PSID")
    if not np.all(np.isnan(z_dpad)):
        ax.plot(
            t_plot,
            _gap(z_dpad),
            color=COLOR_DPAD,
            linewidth=1.2,
            linestyle="--",
            label="DPAD",
        )
    if not np.all(np.isnan(z_varma)):
        ax.plot(
            t_plot,
            _gap(z_varma),
            color=COLOR_VARMA,
            linewidth=1.2,
            linestyle=(0, (4, 1)),
            label="VARMA",
        )

    ax.set_xlabel("trial time (s)")
    ax.set_ylabel("z-score")
    panel_label(
        ax,
        "A",
        (
            f"{session_label} — DBS-{condition_label} — {channel_label}"
            if channel_label
            else f"{session_label} — DBS-{condition_label}"
        ),
    )
    ax.legend()
    return fig


# ── Figs 29-36: Y forecast, EXP_BEHAVIORAL (consecutive OFF/ON pair) ──────────
fig_num = 29
for c2_spec in THESIS_C2_FORECASTS:
    res_p_std, res_v_std, res_v_off_std, res_v_on_std, ch_use = _resolve_varma_off_on(
        c2_spec
    )
    inn = channels_as_str_list(res_p_std.get("input_channels"))
    neu_lbl = inn[ch_use] if ch_use < len(inn) else c2_spec.neural_y_feature_name
    res_p = _load_h5_forecast(results_root, c2_spec.psid_variant) or res_p_std
    fc_v_both = _load_h5_forecast(results_root, c2_spec.varma_variant)
    _voff = (
        c2_spec.varma_variant.replace("dbs_both", "dbs_off")
        if "dbs_both" in c2_spec.varma_variant
        else None
    )
    _von = (
        c2_spec.varma_variant.replace("dbs_both", "dbs_on")
        if "dbs_both" in c2_spec.varma_variant
        else None
    )
    fc_v_off = _load_h5_forecast(results_root, _voff)
    fc_v_on = _load_h5_forecast(results_root, _von)
    fc_d_both = _load_h5_forecast(results_root, c2_spec.dpad_variant)

    # Consecutive OFF/ON pair from PSID forecast parquet
    pair = _consec_pair_fc(res_p, "Y_future_true", "Y_future_pred", ch_use)
    if pair[0] is None:
        print(
            f"Figs {fig_num}-{fig_num+1}: SKIP {c2_spec.section_title} — no consecutive forecast pair"
        )
        fig_num += 2
        continue
    off_i, on_i = pair

    for cond_lbl, cond_stim, i_use, rv_split in (
        ("OFF", "off", off_i, fc_v_off or fc_v_both),
        ("ON", "on", on_i, fc_v_on or fc_v_both),
    ):
        rv_use = rv_split or fc_v_both
        jv_best = (
            _key_index_map(rv_use).get(_trial_key(res_p, i_use)) if rv_use else None
        )
        if jv_best is None:
            print(
                f"Fig {fig_num}: SKIP {c2_spec.section_title} {cond_lbl} — VARMA key not found"
            )
            fig_num += 1
            continue
        rowdata = _forecast_rowdata(
            res_p,
            fc_d_both,
            rv_use,
            i_use,
            ch_use,
            c2_spec,
            sigma_z=None,
            varma_trial_idx=jv_best,
        )
        if rowdata is None:
            print(
                f"Fig {fig_num}: SKIP {c2_spec.section_title} {cond_lbl} — rowdata None"
            )
            fig_num += 1
            continue
        fig = _mpl_forecast_panel(
            rowdata,
            neu_lbl,
            cond_lbl,
            channel_label=neu_lbl,
            session_label=c2_spec.section_title,
        )
        rmse_p = _trial_forecast_rmse(
            res_p, "Y_future_true", "Y_future_pred", i_use, ch_use
        )
        rmse_v = _trial_forecast_rmse(
            rv_use, "Y_future_true", "Y_future_pred", jv_best, ch_use
        )
        fig.savefig(
            str(
                OUT
                / f"fig_{fig_num:03d}_neural_forecast_{c2_spec.section_title}_{cond_lbl}.png"
            )
        )
        plt.show()
        dpad_status = "yes" if fc_d_both is not None else "missing"
        print(
            f"Fig {fig_num}: Y forecast (EXP_BEHAVIORAL) — {c2_spec.section_title} DBS-{cond_lbl} ('{neu_lbl}'). Consecutive pair: OFF={off_i}, ON={on_i}. RMSE: PSID={rmse_p:.3f}, VARMA={rmse_v:.3f}. DPAD={dpad_status}."
        )
        fig_num += 1

In [ ]:
# ── Figs 45-52: Y forecast, EXP_NEURAL (consecutive OFF/ON pair) ───────────────
THESIS_C2_FORECASTS_LAP = []
for _session, (_off, _on) in _TRIAL_INDICES.items():
    _pv, _pt = discover_session_run(results_root, "psid", EXP_NEURAL, _session)
    _dv, _dt = discover_session_run(results_root, "dpad", EXP_NEURAL, _session)
    _vv, _vt = discover_session_run(results_root, "varma", EXP_NEURAL, _session)
    _, _vt_off = discover_session_run(
        results_root, "varma", EXP_NEURAL, _session, "dbs_off"
    )
    _, _vt_on = discover_session_run(
        results_root, "varma", EXP_NEURAL, _session, "dbs_on"
    )
    if not _pt or not _vt:
        continue
    THESIS_C2_FORECASTS_LAP.append(
        ThesisC2ForecastSpec(
            section_title=_session,
            participant_label=_session.split("_")[0],
            psid_variant=_pv,
            dpad_variant=_dv,
            varma_variant=_vv,
            psid_run_ts=_pt,
            dpad_run_ts=_dt,
            varma_run_ts=_vt,
            split="test",
            trial_idx_off=_off,
            trial_idx_on=_on,
            channel_idx=5,
            forecast_target="Y",
            neural_y_feature_name="ECOG_1_beta_27_30_raw",
            varma_run_ts_off=_vt_off or None,
            varma_run_ts_on=_vt_on or None,
        )
    )

fig_num = 45
for c2_spec in THESIS_C2_FORECASTS_LAP:
    res_p_std, _, _, _, ch_use = _resolve_varma_off_on(c2_spec)
    inn = channels_as_str_list(res_p_std.get("input_channels"))
    neu_lbl = inn[ch_use] if ch_use < len(inn) else c2_spec.neural_y_feature_name
    res_p = _load_h5_forecast(results_root, c2_spec.psid_variant) or res_p_std
    fc_v_both = _load_h5_forecast(results_root, c2_spec.varma_variant)
    _voff = (
        c2_spec.varma_variant.replace("dbs_both", "dbs_off")
        if "dbs_both" in c2_spec.varma_variant
        else None
    )
    _von = (
        c2_spec.varma_variant.replace("dbs_both", "dbs_on")
        if "dbs_both" in c2_spec.varma_variant
        else None
    )
    fc_v_off = _load_h5_forecast(results_root, _voff)
    fc_v_on = _load_h5_forecast(results_root, _von)
    fc_d_both = _load_h5_forecast(results_root, c2_spec.dpad_variant)

    pair = _consec_pair_fc(res_p, "Y_future_true", "Y_future_pred", ch_use)
    if pair[0] is None:
        print(
            f"Figs {fig_num}-{fig_num+1}: SKIP {c2_spec.section_title} — no consecutive pair"
        )
        fig_num += 2
        continue
    off_i, on_i = pair

    for cond_lbl, i_use, rv_split in (
        ("OFF", off_i, fc_v_off or fc_v_both),
        ("ON", on_i, fc_v_on or fc_v_both),
    ):
        rv_use = rv_split or fc_v_both
        jv = _key_index_map(rv_use).get(_trial_key(res_p, i_use)) if rv_use else None
        if jv is None:
            print(
                f"Fig {fig_num}: SKIP {c2_spec.section_title} {cond_lbl} — VARMA key missing"
            )
            fig_num += 1
            continue
        rowdata = _forecast_rowdata(
            res_p,
            fc_d_both,
            rv_use,
            i_use,
            ch_use,
            c2_spec,
            sigma_z=None,
            varma_trial_idx=jv,
        )
        if rowdata is None:
            print(
                f"Fig {fig_num}: SKIP {c2_spec.section_title} {cond_lbl} — rowdata None"
            )
            fig_num += 1
            continue
        fig = _mpl_forecast_panel(
            rowdata,
            neu_lbl,
            cond_lbl,
            channel_label=neu_lbl,
            session_label=c2_spec.section_title,
        )
        rmse_p = _trial_forecast_rmse(
            res_p, "Y_future_true", "Y_future_pred", i_use, ch_use
        )
        rmse_v = _trial_forecast_rmse(
            rv_use, "Y_future_true", "Y_future_pred", jv, ch_use
        )
        fig.savefig(
            str(
                OUT
                / f"fig_{fig_num:03d}_y_forecast_lap_{c2_spec.section_title}_{cond_lbl}.png"
            )
        )
        plt.show()
        dpad_status = "yes" if fc_d_both is not None else "missing"
        print(
            f"Fig {fig_num}: Y forecast (EXP_NEURAL) — {c2_spec.section_title} DBS-{cond_lbl} ('{neu_lbl}'). Consecutive pair: OFF={off_i}, ON={on_i}. RMSE: PSID={rmse_p:.3f}, VARMA={rmse_v:.3f}. DPAD={dpad_status}."
        )
        fig_num += 1


# ── Figs 53-60: Z forecast, EXP_NEURAL (Laplacian LFP ch0, consecutive pair) ──
def _z_feat_name_from_fc(fc_p, sess_label, results_root):
    """Get actual Z feature name for ch0 from PSID train parquet."""
    import polars as pl

    try:
        # fc_p variant is like psid_z-as-neural_PDI1_S2_...
        # find matching train parquet
        psid_dirs = sorted(
            (results_root / "psid").glob(
                f"*z-as-neural*{sess_label.replace(' ','_')}*/train"
            )
        )
        if not psid_dirs:
            return "Z ch0"
        pqs = list(psid_dirs[0].glob("test_results_*.parquet"))
        if not pqs:
            return "Z ch0"
        df = pl.read_parquet(str(pqs[0]))
        if "Z_features" not in df.columns:
            return "Z ch0"
        names = df[0]["Z_features"][0].to_list()
        return names[0] if names else "Z ch0"
    except Exception:
        return "Z ch0"


THESIS_C2_FORECASTS_LAP_Z = []
for _session, (_off, _on) in _TRIAL_INDICES.items():
    _pv, _pt = discover_session_run(results_root, "psid", EXP_NEURAL, _session)
    _dv, _dt = discover_session_run(results_root, "dpad", EXP_NEURAL, _session)
    _vv, _vt = discover_session_run(results_root, "varma", EXP_NEURAL, _session)
    _, _vt_off = discover_session_run(
        results_root, "varma", EXP_NEURAL, _session, "dbs_off"
    )
    _, _vt_on = discover_session_run(
        results_root, "varma", EXP_NEURAL, _session, "dbs_on"
    )
    if not _pt or not _vt:
        continue
    THESIS_C2_FORECASTS_LAP_Z.append(
        ThesisC2ForecastSpec(
            section_title=_session,
            participant_label=_session.split("_")[0],
            psid_variant=_pv,
            dpad_variant=_dv,
            varma_variant=_vv,
            psid_run_ts=_pt,
            dpad_run_ts=_dt,
            varma_run_ts=_vt,
            split="test",
            trial_idx_off=_off,
            trial_idx_on=_on,
            channel_idx=0,
            forecast_target="Z",
            neural_y_feature_name="ECOG_1_beta_27_30_raw",
            varma_run_ts_off=_vt_off or None,
            varma_run_ts_on=_vt_on or None,
        )
    )

fig_num = 53
for c2_spec in THESIS_C2_FORECASTS_LAP_Z:
    fc_p = _load_h5_forecast(results_root, c2_spec.psid_variant)
    fc_v_both = _load_h5_forecast(results_root, c2_spec.varma_variant)
    _voff = (
        c2_spec.varma_variant.replace("dbs_both", "dbs_off")
        if "dbs_both" in c2_spec.varma_variant
        else None
    )
    _von = (
        c2_spec.varma_variant.replace("dbs_both", "dbs_on")
        if "dbs_both" in c2_spec.varma_variant
        else None
    )
    fc_v_off = _load_h5_forecast(results_root, _voff)
    fc_v_on = _load_h5_forecast(results_root, _von)
    fc_d_both = _load_h5_forecast(results_root, c2_spec.dpad_variant)

    if fc_p is None:
        print(
            f"Figs {fig_num}-{fig_num+1}: SKIP {c2_spec.section_title} — no PSID Z forecast parquet"
        )
        fig_num += 2
        continue

    z_feat = _z_feat_name_from_fc(fc_p, c2_spec.section_title, results_root)
    pair = _consec_pair_fc(fc_p, "Z_future_true", "Z_future_pred", c2_spec.channel_idx)
    if pair[0] is None:
        print(
            f"Figs {fig_num}-{fig_num+1}: SKIP {c2_spec.section_title} — no consecutive Z forecast pair"
        )
        fig_num += 2
        continue
    off_i, on_i = pair

    for cond_lbl, i_use, rv_split in (
        ("OFF", off_i, fc_v_off or fc_v_both),
        ("ON", on_i, fc_v_on or fc_v_both),
    ):
        rv_use = rv_split or fc_v_both
        jv = _key_index_map(rv_use).get(_trial_key(fc_p, i_use)) if rv_use else None
        if jv is None:
            print(
                f"Fig {fig_num}: SKIP {c2_spec.section_title} {cond_lbl} — VARMA Z key missing"
            )
            fig_num += 1
            continue
        rowdata = _forecast_rowdata(
            fc_p,
            fc_d_both,
            rv_use,
            i_use,
            c2_spec.channel_idx,
            c2_spec,
            sigma_z=None,
            varma_trial_idx=jv,
        )
        if rowdata is None:
            print(
                f"Fig {fig_num}: SKIP {c2_spec.section_title} {cond_lbl} — rowdata None"
            )
            fig_num += 1
            continue
        fig = _mpl_forecast_panel(
            rowdata,
            z_feat,
            cond_lbl,
            channel_label=z_feat,
            session_label=c2_spec.section_title,
        )
        rmse_p = _trial_forecast_rmse(
            fc_p, "Z_future_true", "Z_future_pred", i_use, c2_spec.channel_idx
        )
        rmse_v = _trial_forecast_rmse(
            rv_use, "Z_future_true", "Z_future_pred", jv, c2_spec.channel_idx
        )
        fig.savefig(
            str(
                OUT
                / f"fig_{fig_num:03d}_z_forecast_lap_{c2_spec.section_title}_{cond_lbl}.png"
            )
        )
        plt.show()
        dpad_status = "yes" if fc_d_both is not None else "missing"
        print(
            f"Fig {fig_num}: Z forecast (LFP, EXP_NEURAL) — {c2_spec.section_title} DBS-{cond_lbl} ('{z_feat}'). Consecutive pair: OFF={off_i}, ON={on_i}. RMSE: PSID={rmse_p:.3f}, VARMA={rmse_v:.3f}. DPAD={dpad_status}."
        )
        fig_num += 1

## Figs 096-099: Condition-variant forecast comparison

3-row figure per session: rows = dbs_both / dbs_off / dbs_on model.
Each row shows PSID + DPAD + VARMA forecasts (h2 horizon) vs true Z (velocity X).
Same DBS-ON trial shown across all 3 rows to reveal whether condition-specific training helps.

In [ ]:
from modules.lib.loaders import load_precomputed_results as _lpr


def _load_cond_forecast(
    results_root, model, exp_type, session, condition, horizon="h5"
):
    """Load h5 forecast results for model/condition/session. Returns None if missing."""
    pv, pt = discover_session_run(
        results_root, model.lower(), exp_type, session, condition
    )
    if not pt:
        return None
    fw = pv.split("_", 1)[0]
    fdir = results_root / fw / pv / "forecast" / horizon
    files = sorted((fdir / "test").glob("test_results_*.parquet"))
    if not files:
        return None
    ts = files[0].name[len("test_results_") : -len(".parquet")]
    try:
        return _lpr(fdir, ts, "test")
    except Exception:
        return None


def _pick_median_rmse_trial(fc_res, target_stim, ch_idx=0):
    """Trial index closest to median NRMSE for target_stim."""
    if fc_res is None:
        return 0
    stims = fc_res.get("stim", [])
    Z_true = fc_res.get("Z_future_true", [])
    Z_pred = fc_res.get("Z_future_pred", [])
    pairs = []
    for i, stim in enumerate(stims):
        if normalize_stim(stim) != target_stim or i >= len(Z_true) or i >= len(Z_pred):
            continue
        zt = np.asarray(Z_true[i], dtype=float)
        zp = np.asarray(Z_pred[i], dtype=float)
        zt_ch = zt[:, ch_idx] if zt.ndim == 2 else zt.ravel()
        zp_ch = zp[:, ch_idx] if zp.ndim == 2 else zp.ravel()
        nrmse = float(np.sqrt(np.nanmean((zt_ch - zp_ch) ** 2)))
        if np.isfinite(nrmse):
            pairs.append((nrmse, i))
    if not pairs:
        return 0
    pairs.sort()
    return pairs[len(pairs) // 2][1]


def _draw_cond_row(
    ax, fc_models, ref_fc, ref_trial_idx, ch_idx, row_title, letter, fs=200.0
):
    """Draw one row: context + multi-model Z forecasts + true future.

    ref_fc / ref_trial_idx: the reference parquet (dbs_both PSID) used to look up
    the matching trial in each model's parquet via key matching.
    """
    _STYLES = {
        "PSID": ("-", COLOR_PSID),
        "DPAD": ("--", COLOR_DPAD),
        "VARMA": ((0, (4, 1)), COLOR_VARMA),
    }

    # Key-match ref trial into each model's parquet
    ref_key = _trial_key(ref_fc, ref_trial_idx)
    model_tidx = {}
    for mname, fc in fc_models.items():
        if fc is None:
            model_tidx[mname] = None
            continue
        mp = _key_index_map(fc)
        idx = mp.get(ref_key)
        model_tidx[mname] = idx  # None if trial not present

    # Use first available model as axis reference
    ref_model = next(
        (m for m in ("PSID", "DPAD", "VARMA") if model_tidx.get(m) is not None),
        None,
    )
    if ref_model is None:
        panel_label(ax, letter, row_title)
        ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", va="center")
        return

    ref_tidx = model_tidx[ref_model]
    ref_data = fc_models[ref_model]
    concat = np.asarray(ref_data["Z_concat_for_plot"][ref_tidx], dtype=float)
    ft = np.asarray(ref_data["Z_future_true"][ref_tidx], dtype=float)
    n_total, n_future = concat.shape[0], ft.shape[0]
    n_hist = n_total - n_future

    z_hist = concat[:n_hist, ch_idx] if concat.ndim == 2 else concat[:n_hist]
    z_true_fut = ft[:, ch_idx] if ft.ndim == 2 else ft.ravel()
    t_all = np.arange(n_total) / fs
    t_fut = t_all[n_hist:]

    ax.axvspan(
        t_all[0], t_all[n_hist - 1], color=COLOR_DBS_OFF, alpha=0.06, linewidth=0
    )
    ax.axvspan(t_all[n_hist], t_all[-1], color=COLOR_DBS_ON, alpha=0.06, linewidth=0)
    ax.axvline(t_all[n_hist], color="#555", lw=0.7, ls="--", alpha=0.5)

    ax.plot(t_all[:n_hist], z_hist, color="#aaa", lw=0.8, alpha=0.7, label="history")
    ax.plot(t_fut, z_true_fut, color=COLOR_TRUE, lw=1.6, label="true")

    for mname, fc in fc_models.items():
        tidx = model_tidx[mname]
        if tidx is None or fc is None or tidx >= len(fc.get("Z_future_pred", [])):
            continue
        fp = np.asarray(fc["Z_future_pred"][tidx], dtype=float)
        z_pred = fp[:, ch_idx] if fp.ndim == 2 else fp.ravel()
        ls, col = _STYLES[mname]
        ax.plot(t_fut, z_pred, color=col, lw=1.2, ls=ls, label=mname)

    ax.set_ylabel("z-score")
    panel_label(ax, letter, row_title)


# ── Figs 096-099: Condition-variant forecast comparison (h5) ──────────────────────
_CV_HORIZON = "h5"
_CV_CH = 0  # velocity X
_CV_STIM = "on"  # show DBS-ON trial

fig_num = 96
for sess in SESSIONS:
    both_fc = {
        m: _load_cond_forecast(
            results_root, m, EXP_BEHAVIORAL, sess, "dbs_both", _CV_HORIZON
        )
        for m in ("PSID", "DPAD", "VARMA")
    }
    off_fc = {
        m: _load_cond_forecast(
            results_root, m, EXP_BEHAVIORAL, sess, "dbs_off", _CV_HORIZON
        )
        for m in ("PSID", "DPAD", "VARMA")
    }
    on_fc = {
        m: _load_cond_forecast(
            results_root, m, EXP_BEHAVIORAL, sess, "dbs_on", _CV_HORIZON
        )
        for m in ("PSID", "DPAD", "VARMA")
    }

    for cond_label, fc_dict in [
        ("dbs_both", both_fc),
        ("dbs_off", off_fc),
        ("dbs_on", on_fc),
    ]:
        avail = {m: ("ok" if v is not None else "MISSING") for m, v in fc_dict.items()}
        print(f"  {sess} {cond_label}: {avail}")

    # Select reference trial from PSID dbs_both; key-match into condition parquets
    ref_fc = both_fc["PSID"]
    ref_trial_idx = _pick_median_rmse_trial(ref_fc, _CV_STIM, _CV_CH)

    fig, axes = plt.subplots(3, 1, figsize=(8, 8), layout="constrained", sharex=True)
    _draw_cond_row(
        axes[0], both_fc, ref_fc, ref_trial_idx, _CV_CH, f"{sess} — dbs_both model", "A"
    )
    _draw_cond_row(
        axes[1], off_fc, ref_fc, ref_trial_idx, _CV_CH, f"{sess} — dbs_off model", "B"
    )
    _draw_cond_row(
        axes[2], on_fc, ref_fc, ref_trial_idx, _CV_CH, f"{sess} — dbs_on model", "C"
    )
    axes[0].legend(loc="upper left", fontsize=8)
    axes[2].set_xlabel("time (s)")
    fig.savefig(str(OUT / f"fig_{fig_num:03d}_cond_variant_fc_{sess}.png"))
    plt.show()
    print(
        f"Fig {fig_num}: Condition-variant forecast (h5) — {sess}, DBS-ON trial (ref idx {ref_trial_idx}), "
        f"Z ch0 (velocity X). Rows: dbs_both / dbs_off / dbs_on models."
    )
    fig_num += 1